# F01 GENITOR — PALATIUM
> *Frégate d'Initialisation — Drive Genesis + GLB Scanner*

**Pipeline :**
1. Monter Google Drive
2. Créer la structure DRIVE_PALATIUM
3. Scanner le GLB (maison.glb)
4. Générer `project_scene_config.json` + `tags_draft.json`
5. Valider avec L'Arbitre

---
**Doctrine :** F01 ne lit que `SHARED/` et n'écrit que dans `F01_GENITOR/OUT/`.

## ⚙️ Configuration — À remplir par le Magos

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────
# Chemin racine du projet Drive
DRIVE_ROOT = '/content/drive/MyDrive/DRIVE_PALATIUM'

# Chemin vers maison.glb (doit être dans SHARED/)
GLB_PATH = f'{DRIVE_ROOT}/SHARED/maison.glb'

# Paramètres vidéo
OUTPUT_FORMATS       = ['shorts', 'youtube']  # ['shorts'] ou ['youtube'] ou les deux
FPS                  = 60
DURATION_SHORTS_SEC  = 30
DURATION_YOUTUBE_SEC = 90
# ───────────────────────────────────────────────────────────

## Étape 1 — Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive monté')

## Étape 2 — Installer les dépendances

In [ ]:
!pip install pygltflib -q
print('✅ pygltflib installé')

## Étape 3 — Copier les scripts dans Colab

In [ ]:
## Étape 3 — Télécharger les scripts depuis GitHub
import urllib.request

GITHUB_RAW = 'https://raw.githubusercontent.com/kioka8877-ux/PALATIUM/main/F01_GENITOR/CODEBASE'
scripts = ['pal_f01_genesis.py', 'pal_f01_scanner.py']

for s in scripts:
    url = f'{GITHUB_RAW}/{s}'
    dst = f'/content/{s}'
    urllib.request.urlretrieve(url, dst)
    print(f'✅ Téléchargé : {s}')


## Étape 4 — Créer la structure Drive

In [ ]:
from pal_f01_genesis import create_drive_structure, print_structure_tree

result = create_drive_structure(DRIVE_ROOT)
print_structure_tree(DRIVE_ROOT)

## Étape 5 — Vérifier que maison.glb est présent

In [ ]:
import os

if os.path.exists(GLB_PATH):
    size_mb = os.path.getsize(GLB_PATH) / 1024 / 1024
    print(f'✅ maison.glb trouvé ({size_mb:.1f} MB)')
else:
    print(f'❌ maison.glb introuvable : {GLB_PATH}')
    print('   → Déposez maison.glb dans SHARED/ avant de continuer.')
    raise FileNotFoundError(f'GLB manquant : {GLB_PATH}')

## Étape 6 — Scanner le GLB + Générer les configs

In [ ]:
from pal_f01_scanner import run_f01_pipeline

success = run_f01_pipeline(
    glb_path             = GLB_PATH,
    drive_root           = DRIVE_ROOT,
    output_formats       = OUTPUT_FORMATS,
    fps                  = FPS,
    duration_shorts_sec  = DURATION_SHORTS_SEC,
    duration_youtube_sec = DURATION_YOUTUBE_SEC
)

if not success:
    raise RuntimeError('F01 GENITOR a échoué — voir messages ci-dessus')

## Étape 7 — Afficher les résultats

In [ ]:
import json

OUT_DIR = f'{DRIVE_ROOT}/F01_GENITOR/OUT'

print('=== project_scene_config.json ===')
with open(f'{OUT_DIR}/project_scene_config.json') as f:
    print(json.dumps(json.load(f), indent=2))

print('\n=== tags_draft.json ===')
with open(f'{OUT_DIR}/tags_draft.json') as f:
    tags = json.load(f)
    print(f'  Lampes  : {len(tags["lamps"])}')
    print(f'  Vitres  : {len(tags["windows"])}')
    print(f'  Portes  : {len(tags["doors"])}')
    print(json.dumps(tags, indent=2))

## Étape 8 — Validation avec L'Arbitre (CHECK-OUT)

In [ ]:
# Copier l'Arbitre dans Colab
import shutil
shutil.copy(f'{DRIVE_ROOT}/PAL_ARBITRE.py', '/content/PAL_ARBITRE.py')

!python /content/PAL_ARBITRE.py \
    --frigate F01 \
    --mode check-out \
    --drive-root {DRIVE_ROOT} \
    --verbose

---
## Résumé Transit

Si L'Arbitre valide (CHECK-OUT ✅), copier manuellement :

| Fichier | Source | Destination |
|---------|--------|-------------|
| `project_scene_config.json` | `F01_GENITOR/OUT/` | `F02_OCULUS/IN/`, `F03_SCRIPTORIUM/IN/`, `F04_EDICTA/IN/` |
| `tags_draft.json` | `F01_GENITOR/OUT/` | `F02_OCULUS/IN/` |

Puis copier `maison.glb` depuis `SHARED/` vers :
- `F02_OCULUS/IN/maison.glb`
- `F03_SCRIPTORIUM/IN/maison.glb`

Et les HDRI depuis `SHARED/HDRI/` vers :
- `F02_OCULUS/IN/HDRI/`
- `F03_SCRIPTORIUM/IN/HDRI/`

**Puis inscrire le transfert dans `TRACKING/PALATIUM_TRANSFER_LOG.md`.**

*F01 GENITOR — Scellée. Ad Victoriam.*